# Model Training

Now time to train our predictive models
--

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report

Lets load our engineered data
--

In [ ]:
df=pd.read_csv("/kaggle/input/datasets/nakulhemantkarpe/sentinel-ops-feature-engineered-data/feature_engineered_data.csv")

In [ ]:
df.head()

In [ ]:
x=df.drop("Machine failure",axis=1)
y=df["Machine failure"]

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

In [ ]:
y.value_counts()

since we have very failure count, we will reply on F1 score rather than accuracy
--


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report

Lets make a pipeline to train our models quickly
--

In [ ]:
models = {
    "logistic regression": LogisticRegression(max_iter=2000),
    "decision tree": DecisionTreeClassifier(random_state=42),
    "random forest": RandomForestClassifier(random_state=42),
    "knn": KNeighborsClassifier(),
    "svm": SVC(),
    "xgboost": XGBClassifier(random_state=42,eval_metric="logloss")
}

In [ ]:
for name, model in models.items():

    if name == "xgboost":
        model.fit(x_train.to_numpy(), y_train.to_numpy())
        y_pred = model.predict(x_test.to_numpy())
    else:
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0)
    }

In [ ]:
results_df=pd.DataFrame(results).T
results_df

Models Comparison 
--

- **XGBoost:** Best overall; highest F1-score (0.708) with strong recall (0.656).
- **Random Forest:** Highest accuracy (98.45%) and precision (87.50%), but lower recall than XGBoost.
- **Decision Tree:** Highest recall (75.41%), but lower precision and F1 than XGBoost/RF.
- **Logistic Regression:** High accuracy but relatively low recall (29.51%).
- **KNN:** Weak performance, especially recall (13.11%) and F1 (21.33%).
- **SVM:** Highest precision (100%) but extremely poor recall (1.64%), so it misses most positive cases.
- **Best model currently:** **XGBoost** based on F1-score.
- **Next step:** Compare the **tuned XGBoost, Random Forest, and Decision Tree** results against this baseline.
- **Note:** Accuracy alone is not sufficient because the models show a large difference between accuracy and recall, indicating class imbalance.

# Hyperparameter tuning
 since decision tree and random forest are most promising, we are going to tune them

1 : XGboost
--

In [ ]:


xgb = XGBClassifier(random_state=42, eval_metric="logloss")

xgb_params = {"n_estimators":[100,200,300], "max_depth":[3,5,7], "learning_rate":[0.01,0.05,0.1], "subsample":[0.8,1.0], "colsample_bytree":[0.8,1.0]}

xgb_grid = GridSearchCV(xgb, xgb_params, cv=5, scoring="f1", n_jobs=-1, verbose=1)

xgb_grid.fit(x_train.to_numpy(), y_train.to_numpy())

print("Best Parameters:", xgb_grid.best_params_)
print("Best CV F1:", xgb_grid.best_score_)

In [ ]:
xgb_best = xgb_grid.best_estimator_
y_pred_xgb = xgb_best.predict(x_test.to_numpy())

print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_xgb, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred_xgb, zero_division=0))

Tuned XGBoost 
--

- **Accuracy:** 98.55%
- **Precision:** 83.33%
- **Recall:** 65.57%
- **F1-score:** 73.39%
- **F1-score improved** from 70.80% to 73.39%.
- **Precision improved** from 76.92% to 83.33%.
- **Recall remained unchanged** at 65.57%.
- **Tuned XGBoost is currently the best-performing model.**

2 : Random Forest
--

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf_params = {"n_estimators":[100,200,300], "max_depth":[None,10,20], "min_samples_split":[2,5,10], "min_samples_leaf":[1,2,4]}

rf_grid = GridSearchCV(rf, rf_params, cv=5, scoring="f1", n_jobs=-1, verbose=1)

rf_grid.fit(x_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV F1:", rf_grid.best_score_)

In [ ]:
rf_best = rf_grid.best_estimator_
y_pred_rf = rf_best.predict(x_test)

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred_rf, zero_division=0))

Tuned Random Forest 
--

- **Accuracy:** 98.45%
- **Precision:** 87.50%
- **Recall:** 57.38%
- **F1-score:** 69.31%
- Performance is essentially the same as the baseline Random Forest.
- **F1-score remains below tuned XGBoost (73.39%).**
- Random Forest still has excellent precision but comparatively lower recall.
- **XGBoost remains the leading model.**
- Next: check the **tuned Decision Tree** result before final model selection.

3 : Decision tree
--


In [ ]:
dt = DecisionTreeClassifier(random_state=42)

dt_params = {"max_depth":[None,3,5,10,15,20], "min_samples_split":[2,5,10,20], "min_samples_leaf":[1,2,4,8], "criterion":["gini","entropy"]}

dt_grid = GridSearchCV(dt, dt_params, cv=5, scoring="f1", n_jobs=-1, verbose=1)

dt_grid.fit(x_train, y_train)

print("Best Parameters:", dt_grid.best_params_)
print("Best CV F1:", dt_grid.best_score_)

In [ ]:
dt_best = dt_grid.best_estimator_
y_pred_dt = dt_best.predict(x_test)

print("Accuracy :", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_dt, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred_dt, zero_division=0))

 Tuned Decision Tree 
--

- **Accuracy:** 98.30%
- **Precision:** 72.13%
- **Recall:** 72.13%
- **F1-score:** 72.13%
- Tuning improved the balance between precision and recall.
- **F1-score improved** from 68.66% to 72.13%.
- Recall improved from 75.41% to 72.13%, while precision increased from 63.01% to 72.13%.
- Still slightly below **tuned XGBoost (73.39% F1)**.
- **Final leading model: XGBoost**, with Decision Tree very close.

In [ ]:
predictions = {}

for name, model in models.items():

    if name == "xgboost":
        predictions[name] = model.predict(x_test.to_numpy())
    else:
        predictions[name] = model.predict(x_test)

predictions["Tuned Decision Tree"] = dt_best.predict(x_test)
predictions["Tuned Random Forest"] = rf_best.predict(x_test)
predictions["Tuned XGBoost"] = xgb_best.predict(x_test.to_numpy())

In [ ]:
all_results = pd.DataFrame({
    "Model": list(predictions.keys()),
    "Accuracy": [accuracy_score(y_test, predictions[m]) for m in predictions],
    "Precision": [precision_score(y_test, predictions[m], zero_division=0) for m in predictions],
    "Recall": [recall_score(y_test, predictions[m], zero_division=0) for m in predictions],
    "F1-score": [f1_score(y_test, predictions[m], zero_division=0) for m in predictions]
})

all_results.round(4)

In [ ]:
import matplotlib.pyplot as plt

all_results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="barh",
    figsize=(16, 10)
)

plt.title("Baseline vs Tuned Model Performance", fontsize=20)
plt.xlabel("Score", fontsize=14)
plt.ylabel("Model", fontsize=14)
plt.xlim(0, 1.05)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
xgb_pred=xgb_grid.predict(x_test.to_numpy())

ConfusionMatrixDisplay.from_predictions(y_test,xgb_pred)
plt.title("tuned xgboost confusion matrix")
plt.show()

**tuned xgboost confusion matrix**
--
- The model correctly classified **1931 no-failure cases** and **40 failure cases**.
- It produced **8 false positives** and **21 false negatives**.
- The model detected **40 out of 61 actual failures**, giving a recall of **65.57%**.
- The relatively low number of false positives shows good precision, while the 21 false negatives indicate that some failures are still being missed.

# Final Model Comparison — Notes

- **XGBoost** achieved the best overall baseline performance with an F1-score of **0.708**.
- **Tuned XGBoost** achieved the best overall result with an F1-score of **0.734** and accuracy of **98.55%**.
- **Tuned Decision Tree** reached an F1-score of **0.721**, showing a strong balance between precision and recall.
- **Tuned Random Forest** remained at an F1-score of **0.693**, showing little improvement over its baseline.
- **Random Forest** achieved the highest baseline accuracy (**98.45%**) and precision (**87.50%**).
- **Decision Tree** had the highest baseline recall (**75.41%**).
- **SVM** had 100% precision but extremely low recall (**1.64%**), making it unsuitable despite its high precision.
- **KNN** showed the weakest recall and F1-score among the baseline models.
- **F1-score was used as the primary metric for hyperparameter tuning** because it balances precision and recall.
- **Final selected model: Tuned XGBoost**, based on the highest F1-score among the evaluated models.

In [ ]:
import joblib

joblib.dump(xgb_best, "final_xgboost_model.pkl")

print("Final model saved successfully!")

In [68]:
print(x_train.columns)

Index(['Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_L',
       'Type_M'],
      dtype='object')
